# Vesuvius — Segment Ink Detection (Kaggle)

Self-contained training notebook for PHercParis4 labelled segments.
All outputs land in `/kaggle/working/` and are saved as notebook outputs when the session finishes.

## Kaggle settings (in the notebook sidebar, *before* running)
1. **Accelerator**: T4 GPU, V100, or A100 — **P100 is not supported** (PyTorch 2.2+ requires sm_70+)
2. **Internet**: On  *(required for the initial data download)*
3. **Persistence**: Files only

## What this notebook does
- Installs required packages (`imagecodecs`, etc.)
- Downloads the **4 smallest** labelled segments from `data.aws.ash2txt.org` (~15.7 GB)
- Trains a 33-layer 3D→2D CNN on 3 segments; validates on the smallest (`20231221180251`)
- **Hard training budget: 7.5 h**, then ~1 h reserved for inference + saving
- Saves a checkpoint **after every epoch** — the best model is never lost even if the session is killed
- Ends with a full probability map + visualization for the val segment

## Outputs (`/kaggle/working/`)
- `models/best_segment_model.pth`, `models/last_segment_model.pth`
- `models/training_history.json`, `models/training_curves.png`
- `predictions/{val_seg}_prob.npy`, `.tif`, `_vis.png`

## Cell structure
| Section | Purpose |
|---|---|
| 0 | Install dependencies + GPU check |
| 1 | Paths & config |
| 2 | Download data |
| 3 | Device setup · Data loading · Dataset · Model · Loss |
| 4 | DataLoaders & optimizer · Training loop |
| 5 | Training curves |
| 6 | Inference function · Run inference |
| 7 | Visualization · Output summary |


## 0. Install dependencies & GPU check

In [ ]:
import os, sys, time, gc, math, json, random, ssl, shutil, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np

NOTEBOOK_START = time.time()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IS_KAGGLE = Path('/kaggle/working').exists()
WORK_DIR  = Path('/kaggle/working') if IS_KAGGLE else Path('.').resolve()
DATA_ROOT = WORK_DIR / 'data' / 'labelled_segments'
MODEL_DIR = WORK_DIR / 'models'
PRED_DIR  = WORK_DIR / 'predictions'
for d in (DATA_ROOT, MODEL_DIR, PRED_DIR):
    d.mkdir(parents=True, exist_ok=True)

NUM_LAYERS = 33

# Segments sorted smallest -> largest (surface volume size in GB).
ALL_SEGMENTS = [
    ('20231221180251', 3.5), ('20231031143852', 3.7),
    ('20231016151002', 4.0), ('20231106155351', 4.5),
    ('20230702185753', 4.6), ('20231210121321', 5.1),
]
# Pick 4 smallest => ~15.7 GB + ~115 MB labels. Fits the 20 GB /kaggle/working quota.
SEGMENTS_TO_USE = [s for s, _ in ALL_SEGMENTS[:4]]
VAL_SEGMENT     = SEGMENTS_TO_USE[0]
TRAIN_SEGMENTS  = [s for s in SEGMENTS_TO_USE if s != VAL_SEGMENT]

CFG = dict(
    patch_size      = 256,
    patches_per_seg = 800,
    val_patches     = 96,
    batch_size      = 4,
    grad_accum      = 4,
    num_epochs      = 60,
    lr              = 1e-4,
    weight_decay    = 1e-4,
    ink_pos_thresh  = 0.6,
    ignore_band     = (0.4, 0.6),
    threshold       = 0.4,
    max_train_hours = 7.5,   # hard training budget (from start of training cell)
    seed            = SEED,
)

print('Environment   :', 'Kaggle' if IS_KAGGLE else 'local')
print('Work dir      :', WORK_DIR)
print('Data root     :', DATA_ROOT)
print('Train segments:', TRAIN_SEGMENTS)
print('Val segment   :', VAL_SEGMENT)
print('CFG:', json.dumps(CFG, indent=2, default=str))


## 2. Download data (~15.7 GB, one-time)

Internet must be enabled in the notebook settings.

### 2a. Download helpers

In [ ]:
import subprocess, sys

print('Installing dependencies...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'imagecodecs', 'tifffile'])
print('Done.')

import torch
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    gpu = torch.cuda.get_device_name(0)
    if cap[0] < 7:
        raise RuntimeError(
            f'{gpu} (sm_{cap[0]}{cap[1]}) is not supported by PyTorch 2.2+.\n'
            'Go to Kaggle Settings → Accelerator and select T4 GPU, V100, or A100.'
        )
    print(f'✓  GPU: {gpu}  (CUDA sm_{cap[0]}{cap[1]}) — GPU training enabled.')
else:
    print('No GPU detected — will train on CPU.')


## 1. Setup — paths & config

In [ ]:
BASE_URL   = 'https://data.aws.ash2txt.org/samples/PHercParis4/segments'
VOLUME_DIR = '7.91um-54keV-volume-20230205180739.tifs'
LABEL_FN   = ('PHercParis4-{seg}-7.91um-54keV-volume-20230205180739-'
              '20230702185753-tile64-stride16.tif')

_ctx = ssl.create_default_context()
_ctx.check_hostname = False
_ctx.verify_mode = ssl.CERT_NONE

def _download(url, dest, retries=6, min_bytes=1000):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > min_bytes:
        return True
    tmp = dest.with_suffix(dest.suffix + '.part')
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'vesuvius-train'})
            with urllib.request.urlopen(req, timeout=180, context=_ctx) as r, open(tmp, 'wb') as f:
                while True:
                    chunk = r.read(1 << 20)
                    if not chunk:
                        break
                    f.write(chunk)
            tmp.replace(dest)
            return True
        except Exception as e:
            print('  retry', attempt + 1, '/', retries, '[' + dest.name + ']:', e)
            time.sleep(2 + 2 ** attempt)
    try:
        if tmp.exists():
            tmp.unlink()
    except OSError:
        pass
    return False

def _segment_complete(seg):
    d = DATA_ROOT / seg
    if not (d / 'ink_labels.tif').exists():
        return False
    files = list((d / 'surface_volume').glob('*.tif'))
    return len(files) == NUM_LAYERS and all(p.stat().st_size > 1000 for p in files)

def download_segment(seg):
    d = DATA_ROOT / seg
    print('Downloading', seg)
    ok = _download(
        BASE_URL + '/' + seg + '/ink-detection/' + LABEL_FN.format(seg=seg),
        d / 'ink_labels.tif',
    )
    if not ok:
        print('  FAILED label for', seg)
        return False
    with ThreadPoolExecutor(max_workers=6) as pool:
        futs = []
        for i in range(NUM_LAYERS):
            url = BASE_URL + '/' + seg + '/surface-volumes/' + VOLUME_DIR + '/' + f'{i:02d}.tif'
            dst = d / 'surface_volume' / f'{i:02d}.tif'
            futs.append(pool.submit(_download, url, dst))
        for fut in as_completed(futs):
            fut.result()
    done = _segment_complete(seg)
    print(' ', seg, ':', 'OK' if done else 'INCOMPLETE')
    return done

def _du(p):
    total = 0
    for f in p.rglob('*'):
        if f.is_file():
            total += f.stat().st_size
    return total

print('Download helpers ready.')


### 2b. Run download

In [ ]:
t0 = time.time()
for seg in SEGMENTS_TO_USE:
    if _segment_complete(seg):
        print(seg, 'already on disk')
        continue
    if not download_segment(seg):
        raise RuntimeError(
            'Download failed for ' + seg +
            '. Make sure Internet is enabled in Kaggle notebook settings.'
        )
print('All', len(SEGMENTS_TO_USE), 'segments ready in', round((time.time() - t0) / 60, 1), 'min')
print('Disk used under DATA_ROOT:', round(_du(DATA_ROOT) / 1e9, 2), 'GB')


## 3. Definitions

### 3a. Device setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tifffile as tf
from torch.utils.data import IterableDataset, DataLoader

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = (DEVICE.type == 'cuda')
torch.manual_seed(SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    prop = torch.cuda.get_device_properties(0)
    print('GPU   :', prop.name, '(', round(prop.total_memory / 1e9, 1), 'GB VRAM )')


### 3b. Data loading utilities

In [ ]:
def load_segment_volume(seg_dir: Path) -> np.ndarray:
    layers = []
    for i in range(NUM_LAYERS):
        arr = tf.imread(str(seg_dir / 'surface_volume' / f'{i:02d}.tif'))
        if arr.dtype == np.uint16:
            arr = (arr >> 8).astype(np.uint8)
        elif arr.dtype != np.uint8:
            arr = arr.astype(np.uint8)
        layers.append(arr)
    return np.stack(layers, axis=0)


def load_segment_label(seg_dir: Path) -> np.ndarray:
    lbl = tf.imread(str(seg_dir / 'ink_labels.tif'))
    return lbl.astype(np.float32) / 255.0


def derive_mask(vol: np.ndarray) -> np.ndarray:
    return vol.max(axis=0) > 0


print('Data loading utilities ready.')


### 3c. Dataset — SegmentStreamDataset

In [ ]:
class SegmentStreamDataset(IterableDataset):
    """One segment's volume in RAM at a time; yields patches_per_seg random patches per segment."""

    def __init__(self, data_root, segments, patch_size, patches_per_seg,
                 ink_pos_thresh, augment=True, shuffle=True):
        super().__init__()
        self.data_root      = Path(data_root)
        self.segments       = list(segments)
        self.patch_size     = patch_size
        self.patches_per_seg = patches_per_seg
        self.ink_pos_thresh = ink_pos_thresh
        self.augment        = augment
        self.shuffle        = shuffle

    def _sample_patch(self, vol, lbl, mask):
        H, W = mask.shape
        ps = self.patch_size
        last_y = last_x = 0
        for _ in range(60):
            y = random.randint(0, H - ps)
            x = random.randint(0, W - ps)
            last_y, last_x = y, x
            if mask[y:y + ps, x:x + ps].mean() < 0.6:
                continue
            l_patch = lbl[y:y + ps, x:x + ps]
            if random.random() < 0.5 and l_patch.mean() < self.ink_pos_thresh * 0.25:
                continue
            img = vol[:, y:y + ps, x:x + ps].astype(np.float32) / 255.0
            return img, l_patch.copy(), mask[y:y + ps, x:x + ps].astype(np.float32)
        y, x = last_y, last_x
        img = vol[:, y:y + ps, x:x + ps].astype(np.float32) / 255.0
        return img, lbl[y:y + ps, x:x + ps].copy(), mask[y:y + ps, x:x + ps].astype(np.float32)

    def _augment(self, img, lbl, mask):
        if random.random() < 0.5:
            img  = img[:, :, ::-1].copy()
            lbl  = lbl[:, ::-1].copy()
            mask = mask[:, ::-1].copy()
        if random.random() < 0.5:
            img  = img[:, ::-1, :].copy()
            lbl  = lbl[::-1, :].copy()
            mask = mask[::-1, :].copy()
        k = random.randint(0, 3)
        if k:
            img  = np.rot90(img, k, axes=(1, 2)).copy()
            lbl  = np.rot90(lbl, k).copy()
            mask = np.rot90(mask, k).copy()
        return img, lbl, mask

    def __iter__(self):
        segs = list(self.segments)
        if self.shuffle:
            random.shuffle(segs)
        for seg_id in segs:
            seg_dir = self.data_root / seg_id
            vol  = load_segment_volume(seg_dir)
            lbl  = load_segment_label(seg_dir)
            mask = derive_mask(vol)
            for _ in range(self.patches_per_seg):
                img, y, m = self._sample_patch(vol, lbl, mask)
                if self.augment:
                    img, y, m = self._augment(img, y, m)
                yield torch.from_numpy(img), torch.from_numpy(y), torch.from_numpy(m)
            del vol, lbl, mask
            gc.collect()


print('SegmentStreamDataset defined.')


### 3d. Model — SegmentInkNet

In [ ]:
def _conv3d_block(ci, co):
    return nn.Sequential(
        nn.Conv3d(ci, co, 3, padding=1, bias=False),
        nn.BatchNorm3d(co), nn.ReLU(inplace=True),
        nn.Conv3d(co, co, 3, padding=1, bias=False),
        nn.BatchNorm3d(co), nn.ReLU(inplace=True),
    )


def _conv2d_block(ci, co):
    return nn.Sequential(
        nn.Conv2d(ci, co, 3, padding=1, bias=False),
        nn.BatchNorm2d(co), nn.ReLU(inplace=True),
    )


class SegmentInkNet(nn.Module):
    """3D encoder → 2D decoder. AdaptiveAvgPool3d makes it Z-depth agnostic."""

    def __init__(self):
        super().__init__()
        self.enc1  = _conv3d_block(1, 16)
        self.pool1 = nn.MaxPool3d((2, 1, 1))
        self.enc2  = _conv3d_block(16, 32)
        self.pool2 = nn.MaxPool3d((2, 1, 1))
        self.enc3  = _conv3d_block(32, 64)
        self.zpool = nn.AdaptiveAvgPool3d((1, None, None))
        self.dec1  = _conv2d_block(64, 32)
        self.dec2  = _conv2d_block(32, 16)
        self.head  = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        if x.dim() == 4:
            x = x.unsqueeze(1)
        x = self.pool1(self.enc1(x))
        x = self.pool2(self.enc2(x))
        x = self.enc3(x)
        x = self.zpool(x).squeeze(2)
        x = self.dec1(x)
        x = self.dec2(x)
        return self.head(x)


print('SegmentInkNet defined.')


### 3e. Loss & metrics

In [ ]:
def masked_soft_bce_dice(logits, target, mask, ignore_low=0.4, ignore_high=0.6):
    logits = logits.squeeze(1)
    supervise = mask.bool() & ((target < ignore_low) | (target > ignore_high))
    if supervise.sum() == 0:
        return logits.sum() * 0.0
    bce = F.binary_cross_entropy_with_logits(
        logits[supervise], target[supervise], reduction='mean'
    )
    p = torch.sigmoid(logits)
    m = supervise.float()
    inter = (p * target * m).sum()
    denom = (p * m).sum() + (target * m).sum() + 1e-6
    dice = 1.0 - (2 * inter + 1e-6) / denom
    return 0.5 * bce + 0.5 * dice


@torch.no_grad()
def fbeta(logits, target, mask, beta=0.5, thresh=0.4):
    p = (torch.sigmoid(logits).squeeze(1) > thresh) & mask.bool()
    t = (target > 0.5) & mask.bool()
    tp = (p & t).sum().float()
    fp = (p & ~t).sum().float()
    fn = (~p & t).sum().float()
    prec = tp / (tp + fp + 1e-6)
    rec  = tp / (tp + fn + 1e-6)
    return ((1 + beta ** 2) * prec * rec / (beta ** 2 * prec + rec + 1e-6)).item()


print('Loss and metrics defined.')


## 4. Train (time-budgeted, checkpointed every epoch)

### 4a. DataLoaders & optimizer

In [ ]:
train_ds = SegmentStreamDataset(
    DATA_ROOT, TRAIN_SEGMENTS,
    patch_size=CFG['patch_size'], patches_per_seg=CFG['patches_per_seg'],
    ink_pos_thresh=CFG['ink_pos_thresh'], augment=True, shuffle=True,
)
val_ds = SegmentStreamDataset(
    DATA_ROOT, [VAL_SEGMENT],
    patch_size=CFG['patch_size'], patches_per_seg=CFG['val_patches'],
    ink_pos_thresh=CFG['ink_pos_thresh'], augment=False, shuffle=False,
)
train_loader = DataLoader(
    train_ds, batch_size=CFG['batch_size'],
    num_workers=0, pin_memory=(DEVICE.type == 'cuda'),
)
val_loader = DataLoader(
    val_ds, batch_size=CFG['batch_size'],
    num_workers=0, pin_memory=(DEVICE.type == 'cuda'),
)

model  = SegmentInkNet().to(DEVICE)
opt    = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['num_epochs'])
scaler = torch.amp.GradScaler(enabled=use_amp)

n_params = sum(p.numel() for p in model.parameters())
print('Model:', round(n_params / 1e6, 3), 'M params on', DEVICE)

BEST_PATH = MODEL_DIR / 'best_segment_model.pth'
LAST_PATH = MODEL_DIR / 'last_segment_model.pth'
HIST_PATH = MODEL_DIR / 'training_history.json'

hist        = {'train_loss': [], 'val_loss': [], 'val_f05': [], 'epoch_seconds': [], 'lr': []}
best_f05    = -1.0
train_start = time.time()
budget_s    = CFG['max_train_hours'] * 3600
out_of_time = False

print('DataLoaders, model, and optimizer ready.')


### 4b. Training loop

In [ ]:
for epoch in range(CFG['num_epochs']):
    epoch_start = time.time()
    if epoch_start - train_start > budget_s:
        print('[time budget] elapsed', round((epoch_start - train_start) / 3600, 2),
              'h -- stopping before epoch', epoch + 1)
        out_of_time = True
        break

    # ---- train ----
    model.train()
    loss_sum, steps = 0.0, 0
    opt.zero_grad(set_to_none=True)
    pending = False
    for i, (img, y, m) in enumerate(train_loader):
        img = img.to(DEVICE, non_blocking=True)
        y   = y.to(DEVICE, non_blocking=True)
        m   = m.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=use_amp):
            logits = model(img)
            loss   = masked_soft_bce_dice(
                logits, y, m,
                ignore_low=CFG['ignore_band'][0],
                ignore_high=CFG['ignore_band'][1],
            ) / CFG['grad_accum']
        scaler.scale(loss).backward()
        pending = True
        if (i + 1) % CFG['grad_accum'] == 0:
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            pending = False
        loss_sum += loss.item() * CFG['grad_accum']
        steps += 1
        if steps % 100 == 0:
            remain_min = max(0.0, (budget_s - (time.time() - train_start)) / 60)
            print('  ep', epoch + 1, 'step', steps,
                  'loss', round(loss_sum / steps, 4),
                  'budget_left', round(remain_min, 0), 'min')
        if time.time() - train_start > budget_s:
            print('  [time budget] breaking mid-epoch')
            out_of_time = True
            break
    if pending:
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
    sched.step()
    train_loss = loss_sum / max(steps, 1)

    # ---- val ----
    model.eval()
    v_loss_sum, v_f_sum, v_n = 0.0, 0.0, 0
    with torch.no_grad():
        for img, y, m in val_loader:
            img = img.to(DEVICE, non_blocking=True)
            y   = y.to(DEVICE, non_blocking=True)
            m   = m.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=use_amp):
                logits = model(img)
                v_loss_sum += masked_soft_bce_dice(
                    logits, y, m,
                    ignore_low=CFG['ignore_band'][0],
                    ignore_high=CFG['ignore_band'][1],
                ).item()
            v_f_sum += fbeta(logits, y, m, thresh=CFG['threshold'])
            v_n += 1
    val_loss = v_loss_sum / max(v_n, 1)
    val_f05  = v_f_sum   / max(v_n, 1)

    dt         = time.time() - epoch_start
    current_lr = opt.param_groups[0]['lr']
    hist['train_loss'].append(train_loss)
    hist['val_loss'].append(val_loss)
    hist['val_f05'].append(val_f05)
    hist['epoch_seconds'].append(dt)
    hist['lr'].append(current_lr)
    HIST_PATH.write_text(json.dumps(hist, indent=2))

    ckpt = {
        'model_state_dict': model.state_dict(),
        'epoch'   : epoch + 1,
        'val_f05' : val_f05,
        'val_loss': val_loss,
        'cfg'     : CFG,
    }
    torch.save(ckpt, LAST_PATH)

    mark = ''
    if val_f05 > best_f05:
        best_f05 = val_f05
        torch.save(ckpt, BEST_PATH)
        mark = '  <-- new best'

    print('[epoch', epoch + 1, ']',
          'train', round(train_loss, 4),
          'val',   round(val_loss,   4),
          'F0.5',  round(val_f05,    4),
          'lr', f'{current_lr:.2e}',
          '(', round(dt / 60, 1), 'min )', mark)

    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    if out_of_time:
        break

if not BEST_PATH.exists() and LAST_PATH.exists():
    shutil.copy2(LAST_PATH, BEST_PATH)

total_h = (time.time() - train_start) / 3600
print('')
print('Training done in', round(total_h, 2), 'h.  Best val F0.5 =', round(best_f05, 4))
print('  best checkpoint:', BEST_PATH)
print('  last checkpoint:', LAST_PATH)
print('  history        :', HIST_PATH)


## 5. Training curves

In [ ]:
import matplotlib.pyplot as plt

if hist['train_loss']:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].plot(hist['train_loss'], label='train')
    axes[0].plot(hist['val_loss'],   label='val')
    axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(hist['val_f05'])
    axes[1].set_title('Val F0.5'); axes[1].set_xlabel('epoch'); axes[1].grid(alpha=0.3)
    axes[2].plot(hist['lr'])
    axes[2].set_title('Learning rate'); axes[2].set_xlabel('epoch'); axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(MODEL_DIR / 'training_curves.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved', MODEL_DIR / 'training_curves.png')
else:
    print('No training epochs completed -- cannot plot.')


## 6. Inference on val segment

### 6a. Inference function

In [ ]:
@torch.no_grad()
def predict_segment(model, seg_dir, patch_size=256, stride=128):
    model.eval()
    vol  = load_segment_volume(seg_dir)
    mask = derive_mask(vol)
    H, W = mask.shape
    prob = np.zeros((H, W), dtype=np.float32)
    wsum = np.zeros((H, W), dtype=np.float32)
    vol_t = torch.from_numpy(vol)

    tiles = []
    y_max = max(H - patch_size + 1, 1)
    x_max = max(W - patch_size + 1, 1)
    for y in range(0, y_max, stride):
        for x in range(0, x_max, stride):
            if mask[y:y + patch_size, x:x + patch_size].any():
                tiles.append((y, x))
    print('  inference:', len(tiles), 'tiles over', f'{H}x{W}')

    t0 = time.time()
    for k, (y, x) in enumerate(tiles, 1):
        patch = (vol_t[:, y:y + patch_size, x:x + patch_size]
                 .to(DEVICE).float().div_(255.0).unsqueeze(0))
        with torch.amp.autocast(device_type=DEVICE.type, enabled=use_amp):
            out = torch.sigmoid(model(patch)).squeeze().float().cpu().numpy()
        prob[y:y + patch_size, x:x + patch_size] += out
        wsum[y:y + patch_size, x:x + patch_size] += 1.0
        if k % 200 == 0:
            print('    ', k, '/', len(tiles), f'({round(100 * k / len(tiles))}%)')
    prob = np.where(wsum > 0, prob / np.maximum(wsum, 1e-6), 0.0).astype(np.float32)
    prob *= mask
    del vol, vol_t
    gc.collect()
    print('  inference took', round((time.time() - t0) / 60, 1), 'min')
    return prob


print('predict_segment defined.')


### 6b. Run inference & save predictions

In [ ]:
ckpt = torch.load(BEST_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print('Loaded best checkpoint: epoch', ckpt.get('epoch', '?'),
      '— F0.5', round(ckpt.get('val_f05', -1.0), 4))

seg_id = VAL_SEGMENT
prob   = predict_segment(
    model, DATA_ROOT / seg_id,
    patch_size=CFG['patch_size'], stride=CFG['patch_size'] // 2,
)

np.save(PRED_DIR / f'{seg_id}_prob.npy', prob)
tf.imwrite(PRED_DIR / f'{seg_id}_prob.tif', (prob * 255).astype(np.uint8))
print('Saved prediction for', seg_id, '— shape', prob.shape, '— mean', round(float(prob.mean()), 4))


## 7. Visualization & output summary

### 7a. Visualization

In [ ]:
import matplotlib.pyplot as plt

seg_dir  = DATA_ROOT / seg_id
gt_lbl   = load_segment_label(seg_dir)
ct_arr   = tf.imread(str(seg_dir / 'surface_volume' / f'{NUM_LAYERS // 2:02d}.tif')).astype(np.float32)
ct_mid   = ct_arr / max(float(ct_arr.max()), 1.0)
mask_vis = derive_mask(load_segment_volume(seg_dir))
binary   = (prob > CFG['threshold']).astype(np.uint8) * mask_vis.astype(np.uint8)

ys, xs = np.where(mask_vis)
y0, y1 = int(ys.min()), int(ys.max()) + 1
x0, x1 = int(xs.min()), int(xs.max()) + 1

fig, ax = plt.subplots(2, 3, figsize=(16, 10))
ax[0, 0].imshow(ct_mid[y0:y1, x0:x1], cmap='gray')
ax[0, 0].set_title('CT slice z=' + str(NUM_LAYERS // 2))
ax[0, 1].imshow(prob[y0:y1, x0:x1], cmap='viridis', vmin=0, vmax=1)
ax[0, 1].set_title('Predicted ink probability')
ax[0, 2].imshow(gt_lbl[y0:y1, x0:x1], cmap='gray', vmin=0, vmax=1)
ax[0, 2].set_title('Pseudo-label')
ax[1, 0].imshow(binary[y0:y1, x0:x1], cmap='gray')
ax[1, 0].set_title('Binary t=' + str(CFG['threshold']))
overlay = np.stack([ct_mid[y0:y1, x0:x1]] * 3, axis=-1)
overlay[..., 0] = np.maximum(overlay[..., 0], binary[y0:y1, x0:x1].astype(np.float32))
ax[1, 1].imshow(np.clip(overlay, 0, 1))
ax[1, 1].set_title('Prediction over CT (red)')
ax[1, 2].imshow(np.abs(prob[y0:y1, x0:x1] - gt_lbl[y0:y1, x0:x1]), cmap='hot', vmin=0, vmax=1)
ax[1, 2].set_title('|pred - label|')
for a in ax.ravel():
    a.axis('off')
plt.tight_layout()
vis_path = PRED_DIR / f'{seg_id}_vis.png'
plt.savefig(vis_path, dpi=100, bbox_inches='tight')
plt.show()
print('Saved visualization:', vis_path)


### 7b. Output summary

In [ ]:
print('--- Output files in /kaggle/working ---')
for p in sorted(MODEL_DIR.rglob('*')):
    if p.is_file():
        print('  models/' + str(p.relative_to(MODEL_DIR)),
              round(p.stat().st_size / 1e6, 2), 'MB')
for p in sorted(PRED_DIR.rglob('*')):
    if p.is_file():
        print('  predictions/' + str(p.relative_to(PRED_DIR)),
              round(p.stat().st_size / 1e6, 2), 'MB')

print('')
print('Total wall-clock since notebook start:',
      round((time.time() - NOTEBOOK_START) / 3600, 2), 'h')


## Done

All checkpoints and predictions are under `/kaggle/working/`.
Kaggle will snapshot this directory as the notebook's output when the session/commit finishes, and you can download the whole thing from the notebook's Output tab.

To reload the model locally:
```python
import torch
ckpt = torch.load('best_segment_model.pth', map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
```
